In [ ]:
# =============================================================================
# Functions for Agent Behavior
# =============================================================================

def AskN(i, I, T, c):
    """
    Ask function for NEED: Agent i chooses a product it needs if its inventory is <= 0.
    
    Parameters:
        i (int): Agent index.
        I (np.array): Inventory matrix.
        T (np.array): Memory matrix.
        c (str): Indicator variable (unused input here).
        
    Returns:
        prod_i_want (int): The product the agent wants (or -1 if none needed).
        c (str): Indicator ("N" for need, "H" otherwise).
    """
    need_i = [prod for prod in range(L) if I[i, prod] <= 0]
    if need_i:
        prod_i_want = np.random.choice(need_i)
        c = "N"
    else:
        prod_i_want = -1
        c = "H"
    return prod_i_want, c


def AskG(i, I, T, c):
    """
    Ask function for GREED: Agent i chooses a product based on its memory or randomly.
    
    Parameters:
        i (int): Agent index.
        I (np.array): Inventory matrix.
        T (np.array): Memory matrix.
        c (str): Indicator variable (unused input here).
        
    Returns:
        prod_i_want (int): The product chosen.
        c (str): Always "G" for greed.
    """
    valid_memory = T[i][T[i] != -1] if isinstance(T[i], np.ndarray) else np.array(T[i])[np.array(T[i]) != -1]
    if len(valid_memory) == 0:
        prod_i_want = np.random.randint(0, L)
    else:
        prod_i_want = np.random.choice(valid_memory)
    c = "G"
    return prod_i_want, c


def Ask(i, I, T, c):
    """
    Combined Ask function: Agent i prioritizes need (inventory zero) over greed.
    
    Parameters:
        i (int): Agent index.
        I (np.array): Inventory matrix.
        T (np.array): Memory matrix.
        c (str): Indicator variable (unused input here).
        
    Returns:
        prod_i_want (int): The product chosen.
        c (str): "N" if based on need, "G" if based on greed.
    """
    valid_memory = T[i][T[i] != -1] if isinstance(T[i], np.ndarray) else np.array(T[i])[np.array(T[i]) != -1]
    need_i = [prod for prod in range(L) if I[i, prod] == 0]
    if len(valid_memory) == 0 and not need_i:
        prod_i_want = np.random.randint(0, L)
        c = "G"
    elif need_i:
        prod_i_want = np.random.choice(need_i)
        c = "N"
    else:
        prod_i_want = np.random.choice(valid_memory)
        c = "G"
    return prod_i_want, c


# =============================================================================
# Functions for Trade and Memory Updates
# =============================================================================

def Trade(i, j, prod_i_want, prod_j_want, I, T):
    """
    Performs a trade between agents i and j exchanging products.
    """
    I[i, prod_j_want] -= 1
    I[j, prod_j_want] += 1
    I[j, prod_i_want] -= 1
    I[i, prod_i_want] += 1


def MemoryUpdate(i, j, prod_i_want, prod_j_want, I, T):
    """
    Updates the memory of agents i and j with the traded products.
    """
    memory_index_i = np.random.randint(0, len(T[i]))
    memory_index_j = np.random.randint(0, len(T[j]))
    T[i][memory_index_i] = prod_j_want
    T[j][memory_index_j] = prod_i_want
    


def Pro_Consumption(i, sigma, I, T):
    """
    Handles production and consumption events for agent i based on a probability sigma.
    """
    if np.sqrt(np.random.randn() ** 2) <= sigma and sigma != 0:
        if len(T[i][T[i] != -1]) == 0:
            prod_produce = np.random.randint(0, L)
        else:
            prod_produce = np.random.choice(T[i][T[i] != -1])
        I[i, prod_produce] += 1

        has_i = [prod for prod in range(L) if I[i, prod] != 0]
        prod_consume = np.random.choice(has_i)
        I[i, prod_consume] -= 1


# =============================================================================
# Functions for Selecting Agents and News
# =============================================================================

def pick_agents1D(Amount, cons, Agent_grid):
    """
    Picks two distinct agents randomly (1D).
    """
    return np.random.choice(N, 2, replace=False)


def pick_agents2D(Amount, cons, Agent_grid):
    """
    Picks two distinct agents from a 2D grid with periodic boundaries.
    """
    x, y = np.random.randint(0, w, 2)
    i = Agent_grid[x, y]
    
    while True:
        dx, dy = np.random.choice(range(-cons, cons + 1), 2)
        x2 = (x + dx) % w
        y2 = (y + dy) % w
        j = Agent_grid[x2, y2]
        if i != j:
            return i, j


def weighted_random(M):
    """
    Returns a weighted random integer based on an exponential decay.
    Adjust the decay parameter (L/9) as needed.
    """
    weights = np.exp(-np.arange(1, L + 1) / (L / 9))
    weights /= weights.sum()
    return np.random.choice(np.arange(1, L + 1), p=weights)


def news1D(Amount, sigma, L, Agent_grid, I, T):
    """
    Placeholder for news effect in 1D.
    """
    x = Amount  # Currently does nothing


def news2D(Amount, sigma, L, Agent_grid, I, T):
    """
    Updates the memory of agents in a 2D grid based on weighted news.
    """
    if np.sqrt(np.random.randn() ** 2) <= sigma:
        prod_news = np.random.randint(0, L)
        news_weight = weighted_random(M)

        # Select a random coordinate within the grid
        x1, y1 = np.random.randint(0, round(np.sqrt(Amount)), 2)
        # Define neighbor coordinates with wrapping
        x0, y0 = ((x1 - 1) % w, (y1 - 1) % w)
        x2, y2 = ((x1 + 1) % w, (y1 + 1) % w)
        
        # Collect agents in a 3x3 neighborhood
        a0 = Agent_grid[x0, y0]
        b0 = Agent_grid[x0, y1]
        c0 = Agent_grid[x0, y2]
        a1 = Agent_grid[x1, y0]
        b1 = Agent_grid[x1, y1]
        c1 = Agent_grid[x1, y2]
        a2 = Agent_grid[x2, y0]
        b2 = Agent_grid[x2, y1]
        c2 = Agent_grid[x2, y2]
        neighborhood = [a0, b0, c0, a1, b1, c1, a2, b2, c2]

        for ag in neighborhood:
            for _ in range(news_weight):
                memory_index = np.random.randint(0, M)
                T[ag, memory_index] = prod_news


# =============================================================================
# Utility Functions
# =============================================================================

def random_min_index(lst):
    """
    Chooses a random index among the indices corresponding to the minimum value in a list.
    """
    min_value = min(lst)
    min_indices = [i for i, val in enumerate(lst) if val == min_value]
    return random.choice(min_indices)


def most_prominent(arr):
    """
    Counts the most prominent (frequent) element in each row of arr, ignoring -1.
    Returns a numpy array of the most frequent elements per row.
    """
    result = []
    for row in arr:
        filtered_row = [x for x in row if x != -1]
        if not filtered_row:
            result.append(0)
        else:
            counter = Counter(filtered_row)
            max_count = max(counter.values())
            # Get the first candidate with the maximum count
            candidates = [num for num in filtered_row if counter[num] == max_count]
            result.append(candidates[0])
    return np.array(result)


def Hurst(L_list, T_window):
    """
    Computes an estimate for the Hurst exponent based on a given time window.
    
    Parameters:
        L_list (list or np.array): A list of values.
        T_window (int): The window size for calculation.
        
    Returns:
        float: The mean squared difference over the window.
    """
    Temp = []
    for i, k in enumerate(L_list[:-T_window]):
        c = (L_list[i + T_window] - L_list[i]) ** 2
        Temp.append(c)
    return np.mean(Temp)


def count_elements_in_chunks(data, t_max, L):
    """
    Counts occurrences of each product in chunks of the timeline.
    
    Parameters:
        data (np.array): 1D array with trade data.
        t_max (int): Total number of iterations.
        L (int): Number of products.
        
    Returns:
        np.array: A 2D array with counts per chunk and a time indicator.
    """
    num_chunks = t_max // 50
    counts = np.full((num_chunks, L + 1), np.nan, dtype=float)
    
    for i in range(num_chunks):
        chunk = data[i * 100 : (i + 1) * 100]
        for p in range(L):
            counts[i, p] = (chunk == p).sum()
        counts[i, p + 1] = i * 50
    return counts


def Talk(i, j, I, T):
    """
    Allows agents i and j to discuss a product.
    If both discuss the same product, update their memory.
    """
    if len(T[i][T[i] != -1]) == 0:
        prod_i_talk = np.random.randint(0, L)
    else:
        prod_i_talk = np.random.choice(T[i][T[i] != -1])
    
    if len(T[j][T[j] != -1]) == 0:
        prod_j_talk = np.random.randint(0, L)
    else:
        prod_j_talk = np.random.choice(T[j][T[j] != -1])
    
    if prod_i_talk == prod_j_talk:
        MemoryUpdate(i, j, prod_i_talk, prod_j_talk, I, T)

In [ ]:
# =============================================================================
# Main Simulation Function
# =============================================================================

def Trade_Sim(Start_cond, D, Type):
    """
    Main simulation function for trade among agents.
    
    Parameters:
        Start_cond (tuple): (t_max, N, L, U, M,) The important starting conditions .
        D (int): Dimension flag (2 for 2D, else 1D).
        Type (int): Trade type indicator:
                    2 -> Only Greed,
                    3 -> Only Need,
                    Other -> Normal (both).
                    
    Returns:
        tuple: (I_history, T_history, Frac, Trade_history, Demand_list, Agents_score)
    """
    t_max, N, L, U, M, sigma, w = Start_cond
    M1, M2 = M, M*2
    # Initialize inventory for each agent
    I = np.zeros((N, L), dtype=int)
    # Create a shuffled array of products to distribute equally among agents
    products = np.tile(np.arange(L), N // L)
    products = np.tile(products, U)
    np.random.shuffle(products)
    for idx, prod in enumerate(products):
        agent = idx % N
        I[agent, prod] += 1

    # Initialize memory for each agent (-1 indicates empty memory)
    T = [np.full(M1, -1, dtype=int) if i < 98 else np.full(M2, -1, dtype=int) for i in range(N)]
    
    # Prepare histories and tracking arrays
    I_history = np.empty((t_max, N, L), dtype=int)
    T_history = [None] * t_max
    Demand_list = np.empty((t_max, L))
    Trade_history = -np.ones(t_max * 2, dtype=int)
    Agents_score = np.zeros((t_max, N))
    Trade_index = 0 # Pointer for filling Trade_history

    # Fraction counters: each row is [time_chunk, fraction_greed, fraction_need]
    Frac = np.empty((round(t_max / 100) + 1, 3))
    c = 0
    cG = 0
    cN = 0

    # Determine asking function based on trade type
    if Type == 2:
        Asking = AskG
    elif Type == 3:
        Asking = AskN
    else:
        Asking = Ask

    # Setup agent grid for 2D or 1D simulation
    if D == 2:
        pick_agents = pick_agents2D
        news = news2D
        Agents = list(range(N))
        Agent_grid = np.array(Agents).reshape(w, w)
    else:
        pick_agents = pick_agents1D
        news = news1D
        Agents = list(range(N))
        Agent_grid = np.array(Agents)

    # Main simulation loop
    for trade in range(t_max):
        # Pick two distinct agents
        i, j = pick_agents(N, 1, Agent_grid)
        
        # Optionally include discussion (currently commented out)
        # if np.sqrt(np.random.randn()**2) <= sigma and sigma != 0:
        #     Talk(i, j, I, T)
        
        # Production/Consumption step for both agents
        Pro_Consumption(i, sigma, I, T)
        Pro_Consumption(j, sigma, I, T)
        # Optionally, news can be incorporated:
        # news(N, sigma / 100, L, Agent_grid, I, T)
        
        # Agents decide what product they want based on their strategy
        i_want, c1 = Asking(i, I, T, c)
        j_want, c2 = Asking(j, I, T, c)
        
        # Update counters for Need vs. Greed responses
        cN += (1 if c1 == "N" else 0) + (1 if c2 == "N" else 0)
        cG += (1 if c1 == "G" else 0) + (1 if c2 == "G" else 0)

        if (trade + 1) % 100 == 0:
            Frac[(trade + 1) // 100] = (trade + 1) // 100, cG / 200, cN / 200
            cG = 0
            cN = 0

        # Process the trade if both agents have valid choices and available inventory
        if i_want != -1 and j_want != -1 and I[i, j_want] and I[j, i_want]:
            Trade(i, j, i_want, j_want, I, T)
            '''
            #Correct scoring
            # Update agent scores based on previous demand (note: uses trade-1 index)
            Agents_score[trade, i] = Demand_list[trade - 1, i_want] - Demand_list[trade - 1, j_want]
            Agents_score[trade, j] = Demand_list[trade - 1, j_want] - Demand_list[trade - 1, i_want]
            '''
            #Old scoring
            Agents_score[trade, i] = 1
            Agents_score[trade, j] = 1
            # Record successful trade in history
            Trade_history[Trade_index:Trade_index + 2] = [i_want, j_want]
            Trade_index += 2
        else:
            # Mark trade failure with -1
            Trade_history[Trade_index:Trade_index + 2] = [-1, -1]
            Trade_index += 2

        # Update the demand list (fraction of memory slots holding each product)
        for p in range(L):
            count = sum(np.count_nonzero(T[i] == p) for i in range(N))
            Demand_list[trade, p] = count / sum(len(T[i]) for i in range(N))


        # Cumulative agent score update
        Agents_score[trade, :] += Agents_score[trade - 1, :]
        
        # Update agents' memories based on the trade (or attempted trade)
        MemoryUpdate(i, j, i_want, j_want, I, T)

        # Save current state of inventory and memory
        I_history[trade] = I.copy()
        T_history[trade] = [T[i].copy() for i in range(N)]

    return I_history, T_history, Frac, Trade_history, Demand_list, Agents_score